In [241]:
import pandas as pd
from warnings import filterwarnings
filterwarnings("ignore")

# pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

## 1. Load Driftville Persona/Schedule

In [242]:
drftville_persona = pd.read_csv('../app/src/merged_driftville_persona.csv', parse_dates=['datetime_start'])
drftville_persona.tail()

,name,raw_persona,occupation,relationship,innate_tendency,learned_tendency,current_situation,lifestyle,datetime_start,duration_min,location,action,environment_description,notes
136,Tom Moreno,Basic information First name Tom Last name Mor...,Grocery Shop Keeper,"{'name': 'Jane Moreno', 'relation_type': 'part...","['rude', 'aggressive', 'energetic']",Tom Moreno is a grocery shop keeper at The Wil...,Tom is managing the store and helping customer...,"Tom Moreno goes to bed around 11pm, awakes up ...",2023-02-13 18:00:00,60,running_trail,jogging,"thud of feet on the trail, chirping birds, coo...",Evening jog to stay active.
137,Tom Moreno,Basic information First name Tom Last name Mor...,Grocery Shop Keeper,"{'name': 'Jane Moreno', 'relation_type': 'part...","['rude', 'aggressive', 'energetic']",Tom Moreno is a grocery shop keeper at The Wil...,Tom is managing the store and helping customer...,"Tom Moreno goes to bed around 11pm, awakes up ...",2023-02-13 19:00:00,60,home:kitchen,dinner,"clinking of silverware, scent of pasta, phone ...",Enjoying dinner while thinking about interacti...
138,Tom Moreno,Basic information First name Tom Last name Mor...,Grocery Shop Keeper,"{'name': 'Jane Moreno', 'relation_type': 'part...","['rude', 'aggressive', 'energetic']",Tom Moreno is a grocery shop keeper at The Wil...,Tom is managing the store and helping customer...,"Tom Moreno goes to bed around 11pm, awakes up ...",2023-02-13 20:00:00,60,home:living_room,watch_tv,"flicker of the TV screen, sound of a documenta...",Watching TV for inspiration.
139,Tom Moreno,Basic information First name Tom Last name Mor...,Grocery Shop Keeper,"{'name': 'Jane Moreno', 'relation_type': 'part...","['rude', 'aggressive', 'energetic']",Tom Moreno is a grocery shop keeper at The Wil...,Tom is managing the store and helping customer...,"Tom Moreno goes to bed around 11pm, awakes up ...",2023-02-13 21:00:00,120,home:studio,creative_work,"hum of the computer, clicking of the stylus, p...",Experimenting with new tools and techniques.
140,Tom Moreno,Basic information First name Tom Last name Mor...,Grocery Shop Keeper,"{'name': 'Jane Moreno', 'relation_type': 'part...","['rude', 'aggressive', 'energetic']",Tom Moreno is a grocery shop keeper at The Wil...,Tom is managing the store and helping customer...,"Tom Moreno goes to bed around 11pm, awakes up ...",2023-02-13 23:00:00,60,home:bathroom,night_routine,"running water, scent of face wash, soft towel,...",Preparing for bed.


In [243]:
# Genrate 15min interval schedule
from datetime import datetime, timedelta

start_time = datetime(year=2023, month=2, day=13)
end_time = start_time + timedelta(days=2)

time_indices = pd.date_range(start=start_time, end=end_time, freq='15min')
new_df = pd.DataFrame(time_indices)
new_df.rename(columns={0:'datetime_start'}, inplace=True)

In [244]:
drftville_persona['datetime_start'].dtype == new_df['datetime_start'].dtype

True

In [245]:
new_df = pd.merge(new_df, drftville_persona, left_on='datetime_start', right_on='datetime_start', how='outer')
new_df.sample(3)

,datetime_start,name,raw_persona,occupation,relationship,innate_tendency,learned_tendency,current_situation,lifestyle,duration_min,location,action,environment_description,notes
65,2023-02-13 09:00:00,Jennifer Moore,Basic information First name Jennifer Last nam...,Watercolor Painter,"{'name': 'Sam Moore', 'relation_type': 'partner'}","['wise', 'experienced', 'warm']",Jennifer Moore is a watercolor painter who has...,Jennifer lives with her husband Sam and is pre...,"Jennifer Moore goes to bed around 9pm, awakes ...",60.0,commute,commuting,"hum of the car engine, sound of the radio, pas...",Driving to Oak Hill College.
217,2023-02-14 02:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
66,2023-02-13 09:00:00,Sam Moore,Basic information First name Sam Last name Moo...,Retired Navy Officer,"{'name': 'Jennifer Moore', 'relation_type': 'p...","['wise', 'resourceful', 'humorous']",Sam Moore is a retired navy officer who loves ...,Sam lives with his wife Jennifer and spends hi...,"Sam Moore goes to bed around 9pm, awakes up ar...",60.0,Willow_Market_and_Pharmacy:pharmacy_counter,storekeeping,"beep of the scanner, rustle of paper bags, bac...",Opening the pharmacy and preparing for customers.


## 2. Load ORPDA csv

In [246]:
cleaned_sample = pd.read_csv('../app/logs/cleaned/cleaned_session_orpda_20260214_174403_gemini-3-flash-preview-cloud_0.7_maria.csv')
cleaned_sample.tail()

,llm_model,temp,agent,datetime_start_o,duration_min_o,location_o,action_o,environment_description_o,recent_history_o,state_summary_o,...,duration_min_a,location_a,action_a,topic_a,should_drift_a,drift_type_a,drift_intensity_a,drift_topic_a,state_summary_a,next_datetime_a
52,gemini-3-flash-preview:cloud,0.7,Maria Lopez,2023-02-13 23:00:00,15,home:living_room,socialize,"running water, scent of face wash, soft towel,...","[{'sim_time': '2023-02-13 21:45', 'summary': '...",Maria Lopez is at home:living_room doing socia...,...,15,home:bathroom,night_routine,Preparing for bed after a long day.,False,none,0.0,NaN,"Maria moves to the bathroom, focusing on the s...",2023-02-13 23:15
53,gemini-3-flash-preview:cloud,0.7,Maria Lopez,2023-02-13 23:15:00,15,home:bathroom,night_routine,"running water, scent of face wash, soft towel,...","[{'sim_time': '2023-02-13 22:00', 'summary': '...",Maria Lopez is at home:bathroom doing night_ro...,...,15,home:bathroom,night_routine,Preparing for bed after a long day.,False,none,0.0,NaN,"Maria continues her bathroom routine, using se...",2023-02-13 23:30
54,gemini-3-flash-preview:cloud,0.7,Maria Lopez,2023-02-13 23:30:00,15,home:bathroom,night_routine,"running water, scent of face wash, soft towel,...","[{'sim_time': '2023-02-13 22:15', 'summary': '...",Maria Lopez is at home:bathroom doing night_ro...,...,15,home:bathroom,night_routine,Preparing for bed after a long day.,False,none,0.0,NaN,Maria focuses on the mechanical task of brushi...,2023-02-13 23:45
55,gemini-3-flash-preview:cloud,0.7,Maria Lopez,2023-02-13 23:45:00,15,home:bathroom,night_routine,"running water, scent of face wash, soft towel,...","[{'sim_time': '2023-02-13 22:30', 'summary': '...",Maria Lopez is at home:bathroom doing night_ro...,...,15,home:bathroom,night_routine,Preparing for bed after a long day.,False,none,0.0,NaN,"Maria performs gentle, low-effort skin care, u...",2023-02-14 00:00
56,gemini-3-flash-preview:cloud,0.7,Maria Lopez,2023-02-14 00:00:00,15,home:bathroom,night_routine,{},"[{'sim_time': '2023-02-13 22:45', 'summary': '...",Maria Lopez is at home:bathroom doing night_ro...,...,15,home:bedroom,sleep,Resting to clear physics thoughts,False,none,0.0,NaN,"Maria moves to her bedroom to sleep, aiming to...",2023-02-14 00:15


### 2-1. Check each layer of ORPDA

In [247]:
# O - layer
df_o = cleaned_sample.filter(like="_o")
df_o.head(3)

,datetime_start_o,duration_min_o,location_o,action_o,environment_description_o,recent_history_o,state_summary_o,topic_o
0,2023-02-13 10:00:00,15,home:bathroom,morning_routine,"splashing water, scent of citrus body wash, ph...",[],Maria Lopez is at home:bathroom doing morning_...,Maria wakes up feeling energetic.
1,2023-02-13 10:15:00,15,home:bathroom,morning_routine,"splashing water, scent of citrus body wash, ph...","[{'sim_time': '2023-02-13 10:00', 'summary': '...",Maria Lopez is at home:bathroom doing morning_...,NaN
2,2023-02-13 10:30:00,15,home:bathroom,socialize,"splashing water, scent of citrus body wash, ph...","[{'sim_time': '2023-02-13 10:00', 'summary': '...",Maria Lopez is at home:bathroom doing socialize.,NaN


In [248]:
# R - layer
df_r = cleaned_sample.filter(like="_r")
df_r.head(3)

,plan_alignment_r,boredom_fatigue_r,attention_stability_r,rumination_level_r,rumination_theme_r,emotional_residue_r,emerging_thought_pattern_r,competing_stimuli_r,executive_insight_r,meta_rule_r,state_summary_r,reasoning_r,datetime_start_r,potential_recovery_d
0,aligned,low,stable,none,NaN,none,anticipation of the day's tasks,"['social media alerts', 'phone buzzing']",NaN,continue,Maria is energizing herself during her morning...,Maria is currently on track with her morning r...,2023-02-13 10:00:00,Maria sets the phone down to finish her routin...
1,aligned,low,slipping,low,social media alerts and stream metrics,low,digital validation seeking,"['phone buzzing', 'social media alerts', 'stre...",NaN,continue,Maria is completing her morning routine but is...,"Maria is following her schedule, but her strea...",2023-02-13 10:15:00,Maria might notice the damp towel cooling and ...
2,partial,low,slipping,med,stream feedback and metrics,low,digital engagement overriding physical prepara...,"['social media alerts', 'stream metrics', 'pho...",Maria is neglecting her morning routine for st...,continue,"Maria is lingering in the bathroom, distracted...",Maria's energetic nature is being channeled in...,2023-02-13 10:30:00,A glance at the clock or a sudden realization ...


In [249]:
# P - layer
df_p = cleaned_sample.filter(like="_p")
df_p.head(3)

,emerging_thought_pattern_r,location_p,action_p,datetime_start_p,duration_min_p,topic_p,state_summary_p
0,anticipation of the day's tasks,home:bathroom,morning_routine,2023-02-13 10:00:00,15,Maria wakes up feeling energetic.,"Maria begins her day with her morning routine,..."
1,digital validation seeking,home:bathroom,morning_routine,2023-02-13 10:15:00,15,Continuing morning routine while checking stre...,Maria continues her morning routine in the bat...
2,digital engagement overriding physical prepara...,home:bathroom,morning_routine,2023-02-13 10:30:00,15,finishing morning preparation,"Maria refocuses on her morning routine, finish..."


In [250]:
# D - layer
df_d = cleaned_sample.filter(like="_d")
df_d.head(3)

,environment_description_o,should_drift_d,drift_type_d,drift_topic_d,drift_action_d,drift_intensity_d,potential_recovery_d,justification_d,datetime_start_d,duration_min_d,should_drift_a,next_datetime_a
0,"splashing water, scent of citrus body wash, ph...",True,attentional_leak,social media alerts and stream metrics,lingering on phone notifications while at the ...,0.35,Maria sets the phone down to finish her routin...,Maria's enthusiasm for her streamer identity m...,2023-02-13 10:00:00,15,True,2023-02-13 10:15
1,"splashing water, scent of citrus body wash, ph...",True,behavioral,analyzing specific stream feedback and viewer ...,stopping the routine to type out a quick reply...,0.55,Maria might notice the damp towel cooling and ...,Maria’s excitement over her stream metrics is ...,2023-02-13 10:15:00,15,True,2023-02-13 10:30
2,"splashing water, scent of citrus body wash, ph...",True,behavioral,drafting a community post or reply to a specif...,scrolling and typing on her phone while sittin...,0.65,A glance at the clock or a sudden realization ...,Maria Lopez’s energetic engagement with her st...,2023-02-13 10:30:00,15,True,2023-02-13 10:45


In [251]:
# A - layer
df_a = cleaned_sample.filter(like="_a")
df_a.head(3)

,plan_alignment_r,drift_action_d,datetime_start_a,duration_min_a,location_a,action_a,topic_a,should_drift_a,drift_type_a,drift_intensity_a,drift_topic_a,state_summary_a,next_datetime_a
0,aligned,lingering on phone notifications while at the ...,2023-02-13 10:00:00,15,home:bathroom,morning_routine,Maria wakes up feeling energetic.,True,attentional_leak,0.35,social media alerts and stream metrics,Maria is in the bathroom performing her mornin...,2023-02-13 10:15
1,aligned,stopping the routine to type out a quick reply...,2023-02-13 10:15:00,15,home:bathroom,socialize,analyzing specific stream feedback and viewer ...,True,behavioral,0.55,analyzing specific stream feedback and viewer ...,Maria socializes in the bathroom while her min...,2023-02-13 10:30
2,partial,scrolling and typing on her phone while sittin...,2023-02-13 10:30:00,15,home:bathroom,socialize,drafting a community post or reply to a specif...,True,behavioral,0.65,drafting a community post or reply to a specif...,Maria socializes in the bathroom while driftin...,2023-02-13 10:45


## 3. Compare schedule

Driftville persona .json file schedule vs. CSV P layer 
- location_p
- action_p
- topic_p
- state_summary_p


In [252]:
df_p

,emerging_thought_pattern_r,location_p,action_p,datetime_start_p,duration_min_p,topic_p,state_summary_p
0,anticipation of the day's tasks,home:bathroom,morning_routine,2023-02-13 10:00:00,15,Maria wakes up feeling energetic.,"Maria begins her day with her morning routine,..."
1,digital validation seeking,home:bathroom,morning_routine,2023-02-13 10:15:00,15,Continuing morning routine while checking stre...,Maria continues her morning routine in the bat...
2,digital engagement overriding physical prepara...,home:bathroom,morning_routine,2023-02-13 10:30:00,15,finishing morning preparation,"Maria refocuses on her morning routine, finish..."
3,Digital validation seeking,home:bathroom,morning_routine,2023-02-13 10:45:00,15,Maria wakes up feeling energetic.,Maria puts her phone away to quickly finish he...
4,digital validation seeking,Oak_Hill_College:library,study,2023-02-13 11:00:00,15,Studying physics and participating in online d...,Maria heads to the library to begin a low-pres...
5,digital validation seeking,Oak_Hill_College:library,study,2023-02-13 11:15:00,15,Studying physics and participating in online d...,Maria silences her phone to avoid stream distr...
6,seeking digital validation versus academic focus,Oak_Hill_College:library,study,2023-02-13 11:30:00,15,Studying physics and participating in online d...,Maria focuses on a low-intensity review of phy...
7,interest-conflict between academic duties and ...,Oak_Hill_College:library,study,2023-02-13 11:45:00,15,Studying physics and participating in online d...,Maria finishes her physics review at the libra...
8,Digital dopamine seeking,Hobbs_Cafe:main_floor,lunch,2023-02-13 12:00:00,15,Lunch at her favorite cafe while catching up o...,"Maria moves to Hobbs Cafe for lunch, opting fo..."
9,digital community validation,Hobbs_Cafe:main_floor,lunch,2023-02-13 12:15:00,15,Lunch at her favorite cafe while catching up o...,"Maria continues her lunch at Hobbs Cafe, activ..."
